# Lesson 2.4 - Applied Stats for ML (Iris Classification)

## Objectives
- Train baseline classifier and compute multi-metric evaluation.
- Use confusion matrix and ROC-AUC.
- Explore calibration and bias/variance tradeoff.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibrationDisplay

IRIS_URL = "https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv"


## Section A - Inspect & Prepare Data


In [ ]:
iris_df = pd.read_csv(IRIS_URL)
display(iris_df.head())
display(iris_df.describe())

X = iris_df.drop(columns=["species"])
y = iris_df["species"]


## Section B - Train/Test Split & Scaling


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## Section C - Train a Simple Classifier


In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_s, y_train)
pred = clf.predict(X_test_s)
proba = clf.predict_proba(X_test_s)


## Section D - Evaluate with Multiple Metrics


In [ ]:
acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, average="macro")
rec = recall_score(y_test, pred, average="macro")
f1 = f1_score(y_test, pred, average="macro")
cm = confusion_matrix(y_test, pred)

classes = np.unique(y)
y_test_bin = label_binarize(y_test, classes=classes)
auc_macro = roc_auc_score(y_test_bin, proba, average="macro", multi_class="ovr")

print(acc, prec, rec, f1, auc_macro)
print(cm)


## Section E - Interpret Metrics


In [ ]:
pos_class = classes[0]
y_test_binary = (y_test == pos_class).astype(int)
proba_binary = proba[:, list(classes).index(pos_class)]
CalibrationDisplay.from_predictions(y_test_binary, proba_binary, n_bins=8)
plt.title(f"Calibration for class={pos_class}")
plt.show()


## Section F - Overfitting/Underfitting Demo (Simple)


In [ ]:
simple_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
complex_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
simple_tree.fit(X_train, y_train)
complex_tree.fit(X_train, y_train)

print("simple train/test:", accuracy_score(y_train, simple_tree.predict(X_train)), accuracy_score(y_test, simple_tree.predict(X_test)))
print("complex train/test:", accuracy_score(y_train, complex_tree.predict(X_train)), accuracy_score(y_test, complex_tree.predict(X_test)))


In [ ]:
def assert_minimum_performance(acc, threshold=0.8):
    if acc < threshold:
        raise ValueError(f"Model accuracy {acc:.2f} below threshold {threshold}")

assert_minimum_performance(acc)
print("quality gate passed")


## Business Case Studies & Exceptions
- Accuracy-only selection can fail when business cost of FN/FP differs.
- Poorly calibrated probabilities can break capacity planning.
- Use threshold tuning and segment-wise reporting before deployment.


## Interview Questions & Answers
1. **Q:** Precision vs recall?  
   **A:** Precision controls false positives; recall controls false negatives.
2. **Q:** Why F1?  
   **A:** Balances precision and recall.
3. **Q:** Why not accuracy only?  
   **A:** Can hide class-specific errors.
4. **Q:** Confusion matrix value?  
   **A:** Shows error types.
5. **Q:** ROC-AUC meaning?  
   **A:** Ranking quality over thresholds.
6. **Q:** What is calibration?  
   **A:** Alignment between predicted probabilities and observed outcomes.
7. **Q:** Bias vs variance?  
   **A:** Underfit vs overfit tendency.
8. **Q:** Why strict split discipline?  
   **A:** Prevent leakage.
9. **Q:** Why threshold tuning?  
   **A:** Match metric to business cost.
10. **Q:** What is robust evaluation report?  
    **A:** Multi-metric + confusion matrix + assumptions.
